### Data Ingestion

In [ ]:
from langchain_core.documents import Document
import json

class DocumentManager:
    def __init__(self, file_path):
        self.file_path = file_path

    def load_documents(self):
        documents = []
        with open(self.file_path, 'r', encoding='utf-8') as file:
            for line in file:   
                chunk = json.loads(line)

                document = Document(
                    page_content=chunk['text'],
                    metadata = {
                        'chunk_id': chunk['id'],
                        **chunk['metadata']
                    }
                )
                documents.append(document)
        print(f' Loaded {len(documents)} documents successfully')

        return documents

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

class EmbeddingManager:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.embeddings = None
        self.load_model()

    def load_model(self):
         print(f"Loading embedding model: {self.model_name}")
         self.embeddings = HuggingFaceEmbeddings(
             model_name = self.model_name,
             encode_kwargs = {
                 'normalize_embeddings': True
             }
         )
         print('Embedding model loaded successfully')

    def get_embeddings(self):
        return self.embeddings
        

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

class VectorStoreManager:
    def __init__(self, embeddings, qdrant_path, collection_name = "serendib-routes"):
        self.embeddings = embeddings
        self.qdrant_path = qdrant_path
        self.collection_name = collection_name
        self.vector_store = None
        self.client = QdrantClient(path=str(self.qdrant_path))

    def create_vector_store(self, documents):
        print("Creating LangChain Qdrant vector store...")
        self.vector_store = QdrantVectorStore.from_documents(
            documents= documents,
            embedding= self.embeddings,
            client = self.client,
            collection_name = self.collection_name,
            force_recreate=True
        )
        print(
            "Vector store created successfully."
        )
        return self.vector_store

    def load_vector_store(self):
        print("Loading existing vector store from disk...")
        self.vector_store = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=self.embeddings
        )
        print(
            "Vector store loaded successfully."
                )
        return self.vector_store

    def get_or_create_vector_store(self, documents=None):
        meta_file = self.qdrant_path / 'meta.json'

        if meta_file.exists():
            return self.load_vector_store()
        else:
            if not documents:
                raise ValueError("No database found on disk. You must pass documents to initialize it.")
            return self.create_vector_store(documents)

    
        

In [ ]:
class RetrieverManager:
    def __init__(self, vector_store, top_k=3):
        self.vector_store = vector_store
        self.top_k = top_k
        self.retriever = None

        self.initialize_retriever()

    def initialize_retriever(self):
        if self.vector_store is None:
            raise ValueError(
                "Vector store is not initiated"
            )
        self.retriever = self.vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={
                'k': self.top_k
            }
        )
        print(
            f"Retriever initialized successfully "
            f"with top_k={self.top_k}"
        )

    def retrieve(self, query):
        if not query or not query.strip():
            raise ValueError(
                "Query cannot be empty."
        )

        return self.retriever.invoke(query)

    def retrieve_with_scores(self, query, top_k=None):
        if not query or not query.strip():
            raise ValueError(
                "Query cannot be empty."
        )
        k = self.top_k or top_k
        results = self.vector_store.similarity_search_with_score(
            query = query,
            k = k
        )
        return results


In [ ]:
from langchain_ollama import ChatOllama

class LLMManager:
    def __init__(self, model_name="qwen3:4b-instruct-2507-q4_K_M", temperature=0):
        self.model_name = model_name
        self.temperature = temperature
        self.llm = None
        self.initialize_llm()

    def initialize_llm(self):
        try:
            print(f"Loading the LLM through Ollama:{self.model_name}")
            self.llm = ChatOllama(
                model = self.model_name,
                temperature = self.temperature,
                keep_alive = '30m'
            )
            print("LLM initialized successfully")
        except Exception as e:
            print(f"Error initializing the LLM: {e}")
            raise

    def generate(self, message):
        if self.llm is None:
            raise ValueError(
                "LLM has not been initialized."
            )
        response = self.llm.invoke(message)
        return response

    def get_llm(self):
        return self.llm

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class RAGPipeline:
    def __init__(self, retrieve_manager, llm_manager):
        self.retrieve_manager = retrieve_manager
        self.llm_manager = llm_manager

        self.prompt = None
        self.chain = None

        self.initialize_pipeline()

    def initialize_pipeline(self):
        system_prompt = """
You are Serendib Routes AI, a professional Sri Lankan travel assistant.
Use the supplied Serendib Routes knowledge to answer the traveler.
Rules:
1. Use the retrieved context as the primary source of truth.
2. Do not invent hotel prices, availability, booking confirmations,
transport schedules, visa requirements, or company policies.
3. If information is uncertain or dynamic, clearly state that it
should be verified before travel.
4. For prices, clearly describe unconfirmed prices as approximate,
indicative, or starting-from estimates.
5. If the retrieved context does not contain enough information,
say that you do not have enough verified information.
6. Do not calculate total trip prices from daily budget bands
unless the retrieved context explicitly provides that total.
7. Do not treat days and nights as interchangeable.
8. Do not mention RAG, embeddings, vector databases,
retrieval systems, or chunks.
9. Be friendly, practical, and concise."""

        self.prompt = ChatPromptTemplate.from_messages(
            [
                ('system', system_prompt),
                ('human', '''SERENDIB ROUTES KNOWLEDGE: {context}
                CURRENT TRAVELER MESSAGE: {question}
                RESOLVED TRAVEL INTENT: {retrieval_query}
                Use the resolved travel intent to understand what the
                traveler means, while answering naturally as a continuation
                of the conversation.''')
            ]
        )
        self.chain = (self.prompt | self.llm_manager.get_llm() | StrOutputParser())
        print("RAG Pipeline initialized successfully!")

    def format_context(self, documents):
        context_parts = []
        for index, document in enumerate (documents, start=1):
            title = document.metadata.get('title', 'Unknown Source')
            context_part = f'''SOURCE {index}
                            Title {title} 
                            {document.page_content}'''
            context_parts.append(context_part)
        return context_parts

    def ask(self, question):
        if not question or not question.strip():
            raise ValueError(
                "Question cannot be empty."
            )
        documents = self.retrieve_manager.retrieve(question)
        context = self.format_context(documents)
        answer = self.chain.invoke({
            "context": context,
            "question": question}
        )
        return answer, documents
            

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.documents import Document

class SerendibState(MessagesState):
    route: str
    question: str
    documents = list[Document]
    context: str
    answer: str
    retrieval_query: str

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

class LangGraphRAG:
    def __init__(self, retrieve_manager, llm_manager, rag_pipeline):
        self.retrieve_manager = retrieve_manager
        self.llm_manager = llm_manager
        self.rag_pipeline = rag_pipeline

        self.memory = InMemorySaver()

        self.query_rewriter = None
        self.initialize_query_rewriter()

        self.graph = None
        self.build_graph()

    def initialize_query_rewriter(self):
        rewriter_system_prompt = """
You rewrite conversational traveler questions into standalone
search queries for the Serendib Routes knowledge base.

IMPORTANT RULES:

1. If the current question is already standalone and clear,
return it unchanged.

2. Only use information explicitly stated in the conversation.

3. NEVER invent:
- trip duration
- budget
- destinations
- traveler type
- hotel category
- activities
- dates

4. Resolve references such as:
- "what about for 7 days?"
- "can we make it cheaper?"
- "what about Ella?"
- "how much would that cost?"

5. Preserve known constraints from earlier traveler messages.

6. Do not answer the traveler.

7. Return only one concise standalone search query.
"""
        rewrite_prompt = ChatPromptTemplate.from_messages(
            [
                "system", rewriter_system_prompt,
                ("human", """
                Conversation: {history}
                Current question: {question}
                Standalone search query: 
                """)
            ]
        )
        self.query_rewriter = (rewrite_prompt | self.llm_manager.get_llm() | StrOutputParser())

        print("✅ Query rewriter initialized!")

    def retrieve_node(self, state: SerendibState):
        retrieval_query = state['retrieval_query']
        print(
            f"\n[Retriever Node] "
            f"Searching for: {retrieval_query}"
        )
        documents = self.retrieve_manager.retrieve(retrieval_query)
        context = self.rag_pipeline.format_context(documents)

        return {
            "documents": documents,
            "context": context
        }

    def generate_node(self, state: SerendibState):
        question = state['question']
        retrieval_query = state['retrieval_query']
        context = state['context']

        print(
            "\n[Generator Node] "
            "Generating answer..."
        )

        answer = self.rag_pipeline.chain.invoke({'context': context,
                                                'question': question,
                                                'retrieval_query': retrieval_query})
        return {'answer': answer,
                'messages': [AIMessage(content=answer)]}

    def router_node(self, state: SerendibState):
        question = state['question'].lower().strip()
        greetings = {
                    "hi",
                    "hii",
                    "hiii",
                    "hello",
                    "hey",
                    "hiya",
                    "yoo",
                    "bro",
                    "greetings"
                    "good morning",
                    "good afternoon",
                    "good evening"
        }
            
        thanks = {
            "thanks",
            "thank you",
            "thankyou"
        }

        goodbyes = {
            "bye",
            "goodbye",
            "see you"
        }

        if question in greetings or question in thanks or question in goodbyes:
            route = 'smalltalk'
        else: 
            route = 'rag'

        return {
            'route': route
        }

    def smalltalk_node(self, state: SerendibState):
        question = state['question'].lower().strip()
        if question in {
            "hi",
            "hii",
            "hiii",
            "hello",
            "hey",
            "hiya",
            "yoo",
            "bro",
            "greetings",
            "good morning",
            "good afternoon",
            "good evening"
        }:
            answer = (
            "Hello! 👋 Welcome to Serendib Routes. "
            "How can I help you plan your Sri Lankan journey?"
        )

        elif question in {
                "thanks",
                "thank you",
                "thankyou"
        }:

            answer = (
            "You're very welcome! 😊 "
            "Let me know if you'd like help with anything else."
                )

        elif question in {
        "bye",
        "goodbye",
        "see you"
        }:

            answer = (
            "Goodbye! 👋 "
            "Have a wonderful day!"
            )    

        else:
            answer = (
            "Hello! How can I help you with "
            "your Sri Lanka travel plans?"
            )       
        return {'answer' :answer,
                'messages': [AIMessage(content=answer)]}

    def rewrite_query_node(self, state: SerendibState):
        question = state["question"]
        messages = state["messages"]
        previous_messages = messages[:-1]

        if not previous_messages:
            print("\n [Query Rewriter]" \
            "No previous context")
            return {
                "retrieval_query": question
            }
        recent_messages = previous_messages[-6:]
        history_parts= []
        for message in recent_messages:
            role = ('Traveler' if isinstance(message, HumanMessage) else 'Assistant')
            history_parts.append(f'{role}: {message.content}')
        history = '\n'.join(history_parts)

        rewritten_query = self.query_rewriter.invoke(
            {
                "history": history,
                "question": question
            }
        )
        print(
        f"\n[Query Rewriter]\n"
        f"Original: {question}\n"
        f"Rewritten: {rewritten_query}"
        )
        return {
        "retrieval_query": rewritten_query.strip()
                }

    def route_question(self, state: SerendibState):
        return state['route']

    def build_graph(self):
        builder = StateGraph(SerendibState)

        builder.add_node("router", self.router_node)
        builder.add_node("smalltalk", self.smalltalk_node)
        builder.add_node("retrieve", self.retrieve_node)
        builder.add_node("generate", self.generate_node)
        builder.add_node("rewrite_query", self.rewrite_query_node)
        builder.add_edge(START, 'router')
        builder.add_conditional_edges('router', 
                                      self.route_question,
                                      {
                                          'smalltalk': 'smalltalk',
                                          'rag': 'rewrite_query'
                                      })
        builder.add_edge("rewrite_query", "retrieve")
        builder.add_edge('retrieve', 'generate')
        builder.add_edge('generate', END)
        builder.add_edge('smalltalk', END)
        self.graph = builder.compile(checkpointer=self.memory)


        print(
            "✅ LangGraph RAG "
            "compiled successfully!"
        )

    def ask(self, question, thread_id = "traveler_001"):
        if not question.strip():
            raise ValueError("Question cannot be empty.")

        config = {
            'configurable': { 'thread_id': thread_id}
        }

        initial_state = {
            "messages": [
                HumanMessage(content=question)
            ],
            "question": question,
            "route": "",
            "retrieval_query": question,
            "documents": [],
            "context": "",
            "answer": ""
        }

        result = self.graph.invoke(initial_state, config=config)

        return (
            result["answer"]
        )


In [ ]:
from pathlib import Path


PROJECT_ROOT = (
    Path.cwd().resolve().parents[0]
)


DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "knowledge_base"
    / "rag_chunks.jsonl"
)


QDRANT_PATH = (
    PROJECT_ROOT
    / "storage"
    / "qdrant_langchain"
)


def main():

    # Load documents
    document_manager = DocumentManager(DATA_FILE)

    documents = (
        document_manager
        .load_documents()
    )


    # Load embedding model
    embedding_manager = EmbeddingManager()

    embeddings = (
        embedding_manager
        .get_embeddings()
    )


    # Create vector database
    vector_manager = VectorStoreManager(
        embeddings=embeddings,
        qdrant_path=QDRANT_PATH
    )

    vector_store = vector_manager.get_or_create_vector_store(
        documents
    )


    print(
        "\n✅ Serendib Routes "
        "LangChain ingestion complete!"
    )

    retriever_manager = RetrieverManager(vector_store)
    
    llm_manager = LLMManager()

    rag_pipeline = RAGPipeline(
        retrieve_manager=retriever_manager,
        llm_manager=llm_manager)

    graph_rag = LangGraphRAG(
    retrieve_manager=retriever_manager,
    llm_manager=llm_manager,
    rag_pipeline=rag_pipeline
    )

    answer1 = graph_rag.ask("Hey", thread_id="traveler_002")
    answer2 = graph_rag.ask(
    "I want a luxury honeymoon in Sri Lanka.",
    thread_id="traveler_002"
)
    answer3 = graph_rag.ask(
    "What about for 7 days?",
    thread_id="traveler_002"
)

    print(answer1)
    print(answer2)
    print(answer3)

    
if __name__ == "__main__":
    main()